# Clustering Analysis with Geospatial Data

This notebook analyzes and compares clustering results using different approaches for incorporating polygon information into HDBSCAN clustering. Three scenarios are examined:

1. **Scenario A: Only Original Points** - clustering using point geometries alone
2. **Scenario B: Points + Frequent Areas** - combining points with derived polygon-area data  
3. **Scenario C: Points + Polygon Centroids** - combining points with centroid data

Each method influences cluster size, density parameters, and spatial coverage differently.

## Key Functions Overview

### find_most_frequent_polygon_area(polygons, grid_size_meters=100)
Identifies geographic regions where input polygons overlap most frequently by creating a grid and counting intersections.

### optimize_and_cluster_geometries(geometries, true_lat, n_trials)
Uses HDBSCAN with Optuna optimization to find optimal clustering parameters (min_samples, eps_meters) and returns cluster polygons.

### display_geospatial_dataset(geometries, true_lat, true_lon, cluster_layers)
Visualizes geospatial data and clustering results on interactive Folium maps with toggleable layers.

In [20]:
# Check if running in Google Colab
try:
    from google.colab import drive
    print("Running in Google Colab. Mounting Google Drive...")
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print("Not running in Google Colab. Skipping Google Drive mount.")
    IN_COLAB = False

Not running in Google Colab. Skipping Google Drive mount.


In [21]:
!pip install optuna
!pip install hdbscan
!pip install folium
!pip install shapely
import os
import sys

# Add the parent directory of ng to the Python path for package imports (only needed in Colab)
if IN_COLAB:
    module_path = '/content/drive/MyDrive/'
    if module_path not in sys.path:
        sys.path.append(module_path)

# Basic imports still needed in the main notebook
import hdbscan
import numpy as np
import optuna
import folium
import json
import datetime
from shapely.geometry import mapping # Only mapping, as Point, Polygon, MultiPoint, box are imported within the utils
import importlib

# Define the constant, consistent with previous usage
DEGREES_PER_METER_LAT = 0.0000089

# Import functions from the correct utils package depending on environment
if IN_COLAB:
    # In Colab, use the mounted Drive copy of the ng package
    import ng.utils.haversine_distance as haversine_distance_module
    import ng.utils.extract_points_from_geometries as extract_points_module
    import ng.utils.calculate_median_cluster_radius_meters as median_radius_module
    import ng.utils.cluster_points_and_get_all_cluster_polygons as cluster_polygons_module
    import ng.utils.optimize_and_cluster_geometries as optimize_cluster_module
    import ng.utils.generate_geospatial_dataset as generate_dataset_module
    import ng.utils.display_geospatial_dataset as display_dataset_module
    import ng.utils.find_most_frequent_polygon_area as frequent_area_module
    import ng.utils.convert_polygon_to_points as convert_polygon_module
    import ng.utils.polygons_to_random_points as polygons_to_random_points_module
    import ng.utils.extract_points_from_geometries as extract_points_from_geometries_module
else:
    # When running locally, import from the local utils package
    import utils.haversine_distance as haversine_distance_module
    import utils.extract_points_from_geometries as extract_points_module
    import utils.calculate_median_cluster_radius_meters as median_radius_module
    import utils.cluster_points_and_get_all_cluster_polygons as cluster_polygons_module
    import utils.optimize_and_cluster_geometries as optimize_cluster_module
    import utils.generate_geospatial_dataset as generate_dataset_module
    import utils.display_geospatial_dataset as display_dataset_module
    import utils.find_most_frequent_polygon_area as frequent_area_module
    import utils.convert_polygon_to_points as convert_polygon_module
    import utils.polygons_to_random_points as polygons_to_random_points_module
    import utils.extract_points_from_geometries as extract_points_from_geometries_module

# Reload modules so edits are picked up without restarting the kernel
importlib.reload(haversine_distance_module)
importlib.reload(extract_points_module)
importlib.reload(median_radius_module)
importlib.reload(cluster_polygons_module)
importlib.reload(optimize_cluster_module)
importlib.reload(generate_dataset_module)
importlib.reload(display_dataset_module)
importlib.reload(frequent_area_module)
importlib.reload(convert_polygon_module)
importlib.reload(polygons_to_random_points_module)
importlib.reload(extract_points_from_geometries_module)

print("All functions imported from ng.utils.")

All functions imported from ng.utils.


---
## Step 1: Generate Geospatial Dataset

In [22]:
new_geometries, new_true_lat, new_true_lon = generate_dataset_module.generate_geospatial_dataset()

print(f"Generated new dataset with {len(new_geometries)} geometries around Lat: {new_true_lat:.4f}, Lon: {new_true_lon:.4f}")

Generated True Location: (30.8529, 34.2803)
Calculated Degrees per Meter (Lat): 0.00000890
Calculated Degrees per Meter (Lon): 0.00001037
Generated 100 'most' points (0-20m bell curve).
Generated 20 'diverse' polygons (16 containing true location, 4 non-intersecting).
Generated new dataset with 120 geometries around Lat: 30.8529, Lon: 34.2803


In [23]:
# ============================================================================
# POINT REDUCTION CONFIGURATION
# ============================================================================



# Import the reduce_close_points function
if IN_COLAB:
    import ng.utils.reduce_close_points as reduce_close_points_module
else:
    import utils.reduce_close_points as reduce_close_points_module

importlib.reload(reduce_close_points_module)

# POINT REDUCTION PARAMETERS
use_point_reduction = True  # Set to True to enable point reduction
reduction_min_distance_meters = 5  # Consolidate points within 5 meters

print("\n" + "="*70)
print("POINT REDUCTION SETTINGS")
print("="*70)
print(f"Use point reduction: {use_point_reduction}")
if use_point_reduction:
    print(f"Reduction distance threshold: {reduction_min_distance_meters} meters")
    print("Method: Consolidate points within threshold distance using centroid")
else:
    print("Point reduction disabled - using original points for all scenarios")
print("="*70)



POINT REDUCTION SETTINGS
Use point reduction: True
Reduction distance threshold: 5 meters
Method: Consolidate points within threshold distance using centroid


In [ ]:
from shapely.geometry import Point, Polygon, MultiPoint, box
import random

original_points = extract_points_from_geometries_module.extract_points_from_geometries(new_geometries)
print(f"Extracted {len(original_points)} points from the geometries.")

original_polygons = [geom for geom in new_geometries if isinstance(geom, Polygon)]
print(f"Found {len(original_polygons)} polygons in the dataset.")

# Convert points to coordinate format for clustering
original_points_as_dict = [(pt.x, pt.y) for pt in original_points]

# ===========================================================================
print("\n" + "="*70)
print("SCENARIO 1: Clustering with ORIGINAL POINTS only")
print("="*70)
print(f"Input: {len(original_points_as_dict)} original point(s)")

# SCENARIO 1: DO NOT apply point reduction - use all original points
scenario_1_points = original_points_as_dict
scenario_1_before = len(original_points_as_dict)
print(f"Point reduction DISABLED for Scenario 1 - using all {scenario_1_before} original point(s)")
print(f"(This preserves point density for better cluster detection)")

scenario_1_clusters, scenario_1_radii, scenario_1_cluster_medians = optimize_cluster_module.optimize_and_cluster_geometries(
    scenario_1_points,
    new_true_lat,
    scenario_name="Scenario 1: Original Points Only"
 )

# ===========================================================================
# SCENARIO 2: ALL POLYGONS CONVERTED TO RANDOM POINTS
# ===========================================================================
print("\n" + "="*70)
print("SCENARIO 2: ALL POLYGONS → Random Points with Density Control")
print("="*70)

# FIXED: Use denser point spacing (50m instead of 150m) for better polygon coverage
point_spacing_meters = 50  # Approximate spacing between points in meters (FIXED: was 150m)
print(f"Point spacing parameter: {point_spacing_meters} meters (smaller = more points)")
print(f"  Note: Using 50m spacing to ensure adequate point coverage of all polygons")

# Convert all polygons to a grid of points spaced approximately point_spacing_meters apart
all_polygon_points = polygons_to_random_points_module.polygons_to_random_points(
    original_polygons,
    spacing_meters=point_spacing_meters
)
print(f"Generated {len(all_polygon_points)} random points from {len(original_polygons)} polygon(s)")

# Combine with original points
scenario_2_all_points = original_points_as_dict + [(pt.x, pt.y) for pt in all_polygon_points]
print(f"Total points before center filter: {len(scenario_2_all_points)}")

# Keep Scenario 2 focused around the center (true location)
scenario_2_center_radius_meters = 700
scenario_2_all_points_centered = [
    pt for pt in scenario_2_all_points
    if haversine_distance_module.haversine_distance((pt[1], pt[0]), (new_true_lat, new_true_lon)) <= scenario_2_center_radius_meters
]
scenario_2_before = len(scenario_2_all_points_centered)
print(f"Center filter radius: {scenario_2_center_radius_meters}m")
print(f"Total points after center filter: {scenario_2_before}")

# Point reduction disabled for now
use_point_reduction = False
reduction_min_distance_meters = 5  # Still print for info

if use_point_reduction:
    scenario_2_all_points_reduced = reduce_close_points_module.reduce_close_points(
        scenario_2_all_points_centered,
        reduction_min_distance_meters,
        haversine_distance_module.haversine_distance
    )
    scenario_2_reduced = len(scenario_2_all_points_reduced)
else:
    scenario_2_all_points_reduced = scenario_2_all_points_centered
    scenario_2_reduced = scenario_2_before

print("Number of points before reduction:")
print(f"{scenario_2_before}")
print("Number of points after reduction:")
print(f"{scenario_2_reduced}")
print(f"Reduction enabled: {use_point_reduction}")
if use_point_reduction:
    print(f"Reduction distance threshold: {reduction_min_distance_meters} meters")
print("-----------------------------------")

# ===========================================================================
# SCENARIO 3: HIGH FREQUENCY AREAS → RANDOM POINTS + ORIGINAL POINTS
# ===========================================================================
print("\n" + "="*70)
print("SCENARIO 3: HIGH FREQUENCY Polygon Areas → Random Points + Original Points")
print("="*70)
print("(FIXED: Now includes original points for consistent comparison)")

# Find high frequency polygon areas
high_freq_polygons = frequent_area_module.find_most_frequent_polygon_area(original_polygons, grid_size_meters=100)

# Ensure high_freq_polygons is a list of Polygon(s)
if isinstance(high_freq_polygons, list):
    # Remove any non-Polygon objects (e.g., floats)
    high_freq_polygons = [poly for poly in high_freq_polygons if isinstance(poly, Polygon)]
    print(f"Found {len(high_freq_polygons)} high frequency polygon area(s)")
else:
    if isinstance(high_freq_polygons, Polygon):
        print("Found 1 high frequency polygon area")
        high_freq_polygons = [high_freq_polygons]
    else:
        print("No valid high frequency polygons found.")
        high_freq_polygons = []

# Generate denser random points inside these polygons to improve Scenario 3 clustering
scenario_3_point_spacing_meters = 15  # Denser than Scenario 2
scenario_3_points = polygons_to_random_points_module.polygons_to_random_points(
    high_freq_polygons,
    spacing_meters=scenario_3_point_spacing_meters
)
print(f"Generated {len(scenario_3_points)} random points from high frequency polygons")

# FIXED: Store just high-freq points (will combine with original points in cell-10)
scenario_3_all_points = [(pt.x, pt.y) for pt in scenario_3_points]
scenario_3_before = len(scenario_3_all_points)
print(f"High-frequency points: {scenario_3_before}")
print(f"Original points: {len(original_points)}")
print(f"Combined for Scenario 3: {scenario_3_before + len(original_points)}")

# No reduction for Scenario 3
print("Point reduction DISABLED for Scenario 3")
print("-----------------------------------")

# ===========================================================================
# SCENARIO 4: GEOHASH-BASED POLYGON POINT AGGREGATION
# ===========================================================================
print("\n" + "="*70)
print("SCENARIO 4: GEOHASH-BASED POLYGON POINT AGGREGATION")
print("="*70)

try:
    from geohash2 import encode, decode
    from collections import Counter
    
    GEOHASH_PRECISION = 7  # ~0.15m x 0.15m cells
    TOP_GEOHASH_CELLS = 20  # Keep top N densest cells
    
    print(f"Configuration: geohash precision={GEOHASH_PRECISION}, top {TOP_GEOHASH_CELLS} cells")
    
    # Step 1: Encode all polygon points to geohash
    print(f"\nStep 1: Encoding polygon points to geohash...")
    polygon_geohashes = []
    for pt in all_polygon_points:
        gh = encode(pt.y, pt.x, precision=GEOHASH_PRECISION)
        polygon_geohashes.append(gh)
    
    # Step 2: Count density per geohash cell
    geohash_counts = Counter(polygon_geohashes)
    print(f"  Found {len(geohash_counts)} unique geohash cells")
    
    # Step 3: Get top-N densest cells
    top_cells = geohash_counts.most_common(TOP_GEOHASH_CELLS)
    print(f"\nStep 2: Top {TOP_GEOHASH_CELLS} densest geohash cells:")
    for gh, count in top_cells[:5]:
        lat, lon = decode(gh)
        print(f"  {gh}: {count} points, centered at ({lat:.5f}, {lon:.5f})")
    
    # Step 4: Generate points preferentially from top geohash cells
    scenario_4_points_generated = []
    for gh, count in top_cells:
        lat_center, lon_center = decode(gh)
        # Generate points proportional to density
        num_points_to_generate = min(count // 5, 20)  # Scale down, max 20 per cell
        DEGREES_PER_METER_LAT = 0.0000089
        jitter_deg = DEGREES_PER_METER_LAT * 5  # ~5m jitter
        for _ in range(num_points_to_generate):
            lat_jitter = random.uniform(-jitter_deg, jitter_deg)
            lon_jitter = random.uniform(-jitter_deg, jitter_deg)
            scenario_4_points_generated.append(Point(lon_center + lon_jitter, lat_center + lat_jitter))
    
    print(f"\nStep 3: Generated {len(scenario_4_points_generated)} points from top geohash cells")
    
    # Step 5: Combine with original points
    scenario_4_all_points = original_points_as_dict + [(pt.x, pt.y) for pt in scenario_4_points_generated]
    print(f"  Combined total: {len(scenario_4_all_points)} points (original + geohash-weighted)")
    print("  Geohash library loaded successfully")
    
except ImportError:
    print("WARNING: geohash2 not installed. Installing...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'geohash2'])
    from geohash2 import encode, decode
    from collections import Counter
    print("  geohash2 installed. Please re-run this cell.")
    scenario_4_all_points = original_points_as_dict  # Fallback

print("="*70)

## How to Control Point Density in Scenarios 2 & 3

The **`point_spacing_meters`** parameter controls how densely points are placed inside each polygon by spacing them approximately that far apart (in meters):

- **`point_spacing_meters = 100`** → Sparse distribution (points spaced ~100m apart)
- **`point_spacing_meters = 50`** → Moderate distribution (default, balanced)
- **`point_spacing_meters = 25`** → Dense distribution (points spaced ~25m apart)
- **`point_spacing_meters = 10`** → Very dense distribution (points spaced ~10m apart)

**Change this value** in the "Scenario 2" code cell to see how point spacing affects clustering results!

---
## Step 2: Interactive Map - Compare All Scenarios

Toggle each scenario layer in the map legend (top-right) to see how different polygon-to-point conversion methods affect clustering results.

In [ ]:

# ============================================================================
# FIXED CELL 10: Compare All Scenarios with Proper Configuration
# ============================================================================
print("\n" + "="*70)
print("DEBUG: CLUSTER INSPECTION - FIXED VERSION")
print("="*70)

# CONFIGURATION: Set to True to preserve actual HDBSCAN cluster shapes (convex hulls)
# Set to False to shrink clusters to uniform circles at their centroids (legacy behavior)
PRESERVE_CLUSTER_SHAPES = True

# Number of top clusters to keep per scenario (sorted by distance to true location)
MAX_CLUSTERS_PER_SCENARIO = 3

print(f"\nConfiguration:")
print(f"  PRESERVE_CLUSTER_SHAPES = {PRESERVE_CLUSTER_SHAPES}")
print(f"  MAX_CLUSTERS_PER_SCENARIO = {MAX_CLUSTERS_PER_SCENARIO}")
if PRESERVE_CLUSTER_SHAPES:
    print("  -> Will display actual HDBSCAN cluster polygons (convex hulls)")
else:
    print("  -> Will shrink clusters to uniform circles at centroids")
print("="*70)

print(f"\nScenario 1 clusters: {len(scenario_1_clusters)} cluster(s)")

# Deterministic fallback for red: if optimization finds no cluster, force one.
if len(scenario_1_clusters) == 0:
    print("Scenario 1 fallback: no clusters found, applying deterministic fallback params...")
    scenario_1_fallback_min_samples = 3
    scenario_1_fallback_eps_meters = 25.0
    scenario_1_points_as_shapely = [Point(lon, lat) for lon, lat in scenario_1_points]
    scenario_1_clusters = cluster_polygons_module.cluster_points_and_get_all_cluster_polygons(
        scenario_1_points_as_shapely,
        central_lat=new_true_lat,
        min_samples=scenario_1_fallback_min_samples,
        cluster_selection_epsilon_meters=scenario_1_fallback_eps_meters,
    )

    # Absolute fallback: if still empty, create one tiny circle at nearest original point.
    if len(scenario_1_clusters) == 0 and len(scenario_1_points) > 0:
        nearest_red_point = min(
            scenario_1_points,
            key=lambda pt: haversine_distance_module.haversine_distance(
                (pt[1], pt[0]), (new_true_lat, new_true_lon)
            ),
        )
        scenario_1_clusters = [Point(nearest_red_point[0], nearest_red_point[1]).buffer(2 * DEGREES_PER_METER_LAT)]
        scenario_1_cluster_medians = 2.0
        scenario_1_radii = scenario_1_fallback_min_samples
        print("Scenario 1 absolute fallback applied: created one 2m red circle at nearest point.")

print(f"Scenario 1 clusters after fallback: {len(scenario_1_clusters)} cluster(s)")
for i, cluster in enumerate(scenario_1_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, area={cluster.area:.10f} deg², Bounds={cluster.bounds}")

# Compute average "radius" of red clusters from their bounding boxes (in degrees)
# This is used for uniform circle sizing if PRESERVE_CLUSTER_SHAPES is False
if scenario_1_clusters:
    red_radii_deg = [
        ((c.bounds[2] - c.bounds[0]) + (c.bounds[3] - c.bounds[1])) / 4
        for c in scenario_1_clusters
    ]
    reference_radius_deg = np.mean(red_radii_deg)
else:
    reference_radius_deg = 2 * DEGREES_PER_METER_LAT  # fallback: 2m radius

# Cluster Scenario 2: All Polygons as Random Points
scenario_2_clusters, scenario_2_radii, scenario_2_cluster_medians = optimize_cluster_module.optimize_and_cluster_geometries(
    scenario_2_all_points_reduced,
    new_true_lat,
    scenario_name="Scenario 2: All Polygons as Random Points"
)

# Keep only Scenario 2 clusters near the center (true location)
scenario_2_cluster_center_radius_meters = 700
scenario_2_clusters_within_radius = [
    cluster for cluster in scenario_2_clusters
    if haversine_distance_module.haversine_distance(
        (cluster.centroid.y, cluster.centroid.x),
        (new_true_lat, new_true_lon)
    ) <= scenario_2_cluster_center_radius_meters
]

# Keep top N clusters nearest to the center
scenario_2_clusters = sorted(
    scenario_2_clusters_within_radius,
    key=lambda c: haversine_distance_module.haversine_distance(
        (c.centroid.y, c.centroid.x),
        (new_true_lat, new_true_lon)
    )
)[:MAX_CLUSTERS_PER_SCENARIO]

# Optional: Shrink clusters to uniform circles (only if PRESERVE_CLUSTER_SHAPES is False)
if not PRESERVE_CLUSTER_SHAPES:
    scenario_2_clusters = [c.centroid.buffer(reference_radius_deg) for c in scenario_2_clusters]
    print(f"\nScenario 2 clusters (shrunk to uniform circles): {len(scenario_2_clusters)} cluster(s)")
else:
    print(f"\nScenario 2 clusters (original HDBSCAN shapes): {len(scenario_2_clusters)} cluster(s)")

for i, cluster in enumerate(scenario_2_clusters):
    dist = haversine_distance_module.haversine_distance(
        (cluster.centroid.y, cluster.centroid.x),
        (new_true_lat, new_true_lon)
    )
    print(f"  Cluster {i}: Type={cluster.geom_type}, area={cluster.area:.10f} deg², dist={dist:.1f}m, Bounds={cluster.bounds}")

# ============================================================================
# FIXED: Filter high-frequency areas by proximity to original point centroid
# ============================================================================
print("\n" + "-"*70)
print("Scenario 3: Weight high-frequency areas by proximity to point cloud")
print("-"*70)

# Calculate centroid of original points (clustering uses this as reference, not true location)
original_points_lat = np.mean([pt.y for pt in original_points])
original_points_lon = np.mean([pt.x for pt in original_points])
print(f"  Original point cloud centroid: ({original_points_lat:.5f}, {original_points_lon:.5f})")

# Filter high-frequency polygons to keep only those near the original points
MAX_DISTANCE_FROM_POINT_CLOUD = 200  # meters
filtered_high_freq_polygons = []
filtered_out_count = 0

for poly in high_freq_polygons:
    dist_to_point_cloud = haversine_distance_module.haversine_distance(
        (poly.centroid.y, poly.centroid.x),
        (original_points_lat, original_points_lon)
    )
    if dist_to_point_cloud <= MAX_DISTANCE_FROM_POINT_CLOUD:
        filtered_high_freq_polygons.append(poly)
    else:
        filtered_out_count += 1
        print(f"    Filtered out: high-freq area at ({poly.centroid.y:.5f}, {poly.centroid.x:.5f}) - {dist_to_point_cloud:.1f}m from point cloud")

if filtered_high_freq_polygons:
    print(f"  Kept {len(filtered_high_freq_polygons)}/{len(high_freq_polygons)} high-frequency areas")
    if filtered_out_count > 0:
        print(f"  Filtered out {filtered_out_count} areas > {MAX_DISTANCE_FROM_POINT_CLOUD}m from point cloud")
    high_freq_polygons = filtered_high_freq_polygons
else:
    print(f"  WARNING: No high-frequency areas within {MAX_DISTANCE_FROM_POINT_CLOUD}m of point cloud")
    print("  Using all {len(high_freq_polygons)} high-frequency areas anyway")

# Regenerate Scenario 3 points from filtered high-freq polygons
scenario_3_points = polygons_to_random_points_module.polygons_to_random_points(
    high_freq_polygons,
    spacing_meters=scenario_3_point_spacing_meters
)
print(f"  Regenerated {len(scenario_3_points)} points from filtered high-frequency areas")
scenario_3_all_points = [(pt.x, pt.y) for pt in scenario_3_points]

# Cluster Scenario 3: High Frequency Areas + Original Points
scenario_3_combined_points = scenario_3_all_points + [(pt.x, pt.y) for pt in original_points]
print(f"  High-frequency points: {len(scenario_3_all_points)}")
print(f"  Original points: {len(original_points)}")
print(f"  Combined total: {len(scenario_3_combined_points)}")

scenario_3_clusters, scenario_3_radii, scenario_3_cluster_medians = optimize_cluster_module.optimize_and_cluster_geometries(
    scenario_3_combined_points,
    new_true_lat,
    scenario_name="Scenario 3: High Frequency Areas + Original Points (filtered by proximity)",
    min_samples_range=(5, 15),
    eps_meters_range=(5.0, 20.0)
)

# Keep top N clusters nearest to the center
scenario_3_clusters = sorted(
    scenario_3_clusters,
    key=lambda c: haversine_distance_module.haversine_distance(
        (c.centroid.y, c.centroid.x),
        (new_true_lat, new_true_lon)
    )
)[:MAX_CLUSTERS_PER_SCENARIO]

# Optional: Shrink clusters to uniform circles (only if PRESERVE_CLUSTER_SHAPES is False)
if not PRESERVE_CLUSTER_SHAPES:
    scenario_3_clusters = [c.centroid.buffer(reference_radius_deg) for c in scenario_3_clusters]
    reference_radius_m = reference_radius_deg / DEGREES_PER_METER_LAT
    print(f"\nScenario 3 clusters (shrunk to uniform circles): {len(scenario_3_clusters)} cluster(s), radius = {reference_radius_m:.2f}m")
else:
    print(f"\nScenario 3 clusters (original HDBSCAN shapes): {len(scenario_3_clusters)} cluster(s)")

for i, cluster in enumerate(scenario_3_clusters):
    dist = haversine_distance_module.haversine_distance(
        (cluster.centroid.y, cluster.centroid.x),
        (new_true_lat, new_true_lon)
    )
    print(f"  Cluster {i}: Type={cluster.geom_type}, area={cluster.area:.10f} deg², dist={dist:.1f}m, Bounds={cluster.bounds}")

# ============================================================================
# SCENARIO 4: GEOHASH-BASED CLUSTERING
# ============================================================================
print("\n" + "-"*70)
print("Scenario 4: Geohash-Based Polygon Aggregation")
print("-"*70)

scenario_4_clusters, scenario_4_radii, scenario_4_cluster_medians = optimize_cluster_module.optimize_and_cluster_geometries(
    scenario_4_all_points,
    new_true_lat,
    scenario_name="Scenario 4: Geohash-Based Aggregation",
    min_samples_range=(5, 15),
    eps_meters_range=(5.0, 25.0)
)

# Keep top N clusters nearest to the center
scenario_4_clusters = sorted(
    scenario_4_clusters,
    key=lambda c: haversine_distance_module.haversine_distance(
        (c.centroid.y, c.centroid.x),
        (new_true_lat, new_true_lon)
    )
)[:MAX_CLUSTERS_PER_SCENARIO]

# Optional: Shrink clusters to uniform circles (only if PRESERVE_CLUSTER_SHAPES is False)
if not PRESERVE_CLUSTER_SHAPES:
    scenario_4_clusters = [c.centroid.buffer(reference_radius_deg) for c in scenario_4_clusters]
    print(f"\nScenario 4 clusters (shrunk to uniform circles): {len(scenario_4_clusters)} cluster(s)")
else:
    print(f"\nScenario 4 clusters (original HDBSCAN shapes): {len(scenario_4_clusters)} cluster(s)")

for i, cluster in enumerate(scenario_4_clusters):
    dist = haversine_distance_module.haversine_distance(
        (cluster.centroid.y, cluster.centroid.x),
        (new_true_lat, new_true_lon)
    )
    print(f"  Cluster {i}: Type={cluster.geom_type}, area={cluster.area:.10f} deg², dist={dist:.1f}m, Bounds={cluster.bounds}")

# Final validation
if len(scenario_1_clusters) == 0 and len(scenario_2_clusters) == 0 and len(scenario_3_clusters) == 0 and len(scenario_4_clusters) == 0:
    print("\nWARNING: NO CLUSTERS FOUND!")
    print("This likely means HDBSCAN found no significant clusters in any scenario.")
    print("Possible reasons:")
    print("  1. Point density is too low")
    print("  2. Clustering parameters (min_samples, eps_meters) are too restrictive")
    print("  3. Data points are too scattered/isolated")
elif len(scenario_1_clusters) > 0 or len(scenario_2_clusters) > 0 or len(scenario_3_clusters) > 0 or len(scenario_4_clusters) > 0:
    print("\n" + "="*70)
    print("SUMMARY: All scenarios completed successfully")
    print(f"  Scenario 1: {len(scenario_1_clusters)} cluster(s)")
    print(f"  Scenario 2: {len(scenario_2_clusters)} cluster(s)")
    print(f"  Scenario 3: {len(scenario_3_clusters)} cluster(s) - filtered by proximity to point cloud")
    print(f"  Scenario 4: {len(scenario_4_clusters)} cluster(s) - geohash-based aggregation")
    print("="*70)
print("\n")


In [26]:
# Diagnostic: Print cluster coordinates and bounds for visual inspection
print("\n" + "="*70)
print("DIAGNOSTIC: Cluster Geometry Details")
print("="*70)
print(f"True location: Lat={new_true_lat}, Lon={new_true_lon}")

print(f"\nScenario 1 clusters: {len(scenario_1_clusters)}")
for i, cluster in enumerate(scenario_1_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, Bounds={cluster.bounds}")
    if cluster.geom_type == 'Polygon':
        coords = list(cluster.exterior.coords)
        print(f"    First 3 coords: {coords[:3]}")
        print(f"    All coords: {coords}")

print(f"\nScenario 2 clusters: {len(scenario_2_clusters)}")
for i, cluster in enumerate(scenario_2_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, Bounds={cluster.bounds}")
    if cluster.geom_type == 'Polygon':
        coords = list(cluster.exterior.coords)
        print(f"    First 3 coords: {coords[:3]}")
        print(f"    All coords: {coords}")

print(f"\nScenario 3 clusters: {len(scenario_3_clusters)}")
for i, cluster in enumerate(scenario_3_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, Bounds={cluster.bounds}")
    if cluster.geom_type == 'Polygon':
        coords = list(cluster.exterior.coords)
        print(f"    First 3 coords: {coords[:3]}")
        print(f"    All coords: {coords}")

print("="*70 + "\n")


DIAGNOSTIC: Cluster Geometry Details
True location: Lat=30.85288763675528, Lon=34.28026538043312

Scenario 1 clusters: 0

Scenario 2 clusters: 6
  Cluster 0: Type=Polygon, Bounds=(34.28034895240353, 30.852773653836582, 34.28038455240353, 30.85280925383658)
    First 3 coords: [(34.28038455240353, 30.85279145383658), (34.280384466691665, 30.852789709131482), (34.28038421038152, 30.85278798122885)]
    All coords: [(34.28038455240353, 30.85279145383658), (34.280384466691665, 30.852789709131482), (34.28038421038152, 30.85278798122885), (34.28038378594151, 30.852786286769327), (34.28038319745921, 30.852784642071484), (34.28038245060203, 30.852783062974666), (34.28038155256263, 30.85278156468643), (34.2803805119896, 30.852780161636122), (34.28037933890423, 30.852778867335875), (34.280378044603985, 30.85277769425051), (34.28037664155368, 30.85277665367748), (34.280375143265445, 30.852775755638074), (34.28037356416863, 30.852775008780903), (34.28037191947079, 30.852774420298605), (34.2803702

In [ ]:
print("\n" + "="*70)
print("INTERACTIVE MAP - SCENARIOS 1, 2, 3, 4")
print("="*70)
print("Displaying clustering results on an interactive map.\n")
print("Layer Legend:")
print("  📍 Gray: Base Dataset (Original polygons)")
print("  🔴 Red: Scenario 1 - Original Points Only")
print("  🟢 Green: Scenario 2 - All Polygons as Random Points")
print("  🔵 Blue: Scenario 3 - Clustered High Frequency Areas")
print("  🩷 Pink: Scenario 4 - Geohash-Based Aggregation")
print("  🟣 Purple: Raw High Frequency Polygons")
print("\nTip: Toggle layers in the legend (top-right) to compare scenarios.")
print("="*70 + "\n")

centroid_polygons = new_geometries
print(
    f"Creating map with {len(scenario_1_clusters)} clusters from Scenario 1, "
    f"{len(scenario_2_clusters)} clusters from Scenario 2, "
    f"{len(scenario_3_clusters)} clusters from Scenario 3, "
    f"{len(scenario_4_clusters)} clusters from Scenario 4, and "
    f"{len(high_freq_polygons)} raw high frequency polygons..."
)

# Update display function to include 4th scenario - we need to modify the call
# The display_geospatial_dataset function expects cluster_list, median_list, etc.
# Let me check if we need to update the function call

map_display = display_dataset_module.display_geospatial_dataset(
    [scenario_1_clusters, scenario_2_clusters, scenario_3_clusters, scenario_4_clusters, high_freq_polygons],
    [scenario_1_cluster_medians, scenario_2_cluster_medians, scenario_3_cluster_medians, scenario_4_cluster_medians, None],
    centroid_polygons,
    new_true_lat,
    new_true_lon
)

# Fallback output in case notebook webview rendering is blank.
map_html_path = "clustering_comparison_map.html"
map_display.save(map_html_path)
print(f"Map created successfully! Saved HTML to: {map_html_path}")

from IPython.display import display
display(map_display)

In [28]:
from utils.haversine_distance import haversine_distance
from shapely.geometry import Point
import numpy as np

# Calculate centroids for clusters
red_centroids = [cluster.centroid for cluster in scenario_1_clusters if hasattr(cluster, 'centroid')]
green_centroids = [cluster.centroid for cluster in scenario_2_clusters if hasattr(cluster, 'centroid')]
blue_centroids = [cluster.centroid for cluster in scenario_3_clusters if hasattr(cluster, 'centroid')]

# True location as Point
yellow_point = Point(new_true_lon, new_true_lat)

# Calculate distances from centroids to true location
red_distances = [haversine_distance((c.y, c.x), (new_true_lat, new_true_lon)) for c in red_centroids]
green_distances = [haversine_distance((c.y, c.x), (new_true_lat, new_true_lon)) for c in green_centroids]
blue_distances = [haversine_distance((c.y, c.x), (new_true_lat, new_true_lon)) for c in blue_centroids]

print("\n--- Cluster Distance Statistics ---")
print(f"Red cluster distances to true location: {red_distances}")
print(f"Green cluster distances to true location: {green_distances}")
print(f"Blue cluster distances to true location: {blue_distances}")
if red_distances:
    print(f"Red cluster mean distance: {np.mean(red_distances):.2f} m, std: {np.std(red_distances):.2f} m")
if green_distances:
    print(f"Green cluster mean distance: {np.mean(green_distances):.2f} m, std: {np.std(green_distances):.2f} m")
if blue_distances:
    print(f"Blue cluster mean distance: {np.mean(blue_distances):.2f} m, std: {np.std(blue_distances):.2f} m")
print("-----------------------------------\n")


--- Cluster Distance Statistics ---
Red cluster distances to true location: []
Green cluster distances to true location: [14.423132617163619, 374.8660546360322, 258.71215842551203, 120.0578549929837, 6.6279002441038175, 122.06144276085054]
Blue cluster distances to true location: [24.468075130155203, 58.05612007129476, 71.54809449695107]
Green cluster mean distance: 149.46 m, std: 131.02 m
Blue cluster mean distance: 51.36 m, std: 19.80 m
-----------------------------------



In [29]:
print('--- Scenario 3 quick diagnostics ---')
print('raw_high_freq_polygons:', len(high_freq_polygons))
print('scenario_3_points:', len(scenario_3_points))
print('scenario_3_all_points:', len(scenario_3_all_points))
print('scenario_3_clusters:', len(scenario_3_clusters))
print('scenario_3_cluster_medians:', scenario_3_cluster_medians)
print('scenario_3_radii:', scenario_3_radii)

--- Scenario 3 quick diagnostics ---
raw_high_freq_polygons: 18
scenario_3_points: 648
scenario_3_all_points: 648
scenario_3_clusters: 3
scenario_3_cluster_medians: 19.360330002285522
scenario_3_radii: 7


In [30]:

# Deep diagnostic: compare high_freq_polygons vs scenario_3_clusters
print("=== HIGH FREQ POLYGONS (Purple) ===")
print(f"Count: {len(high_freq_polygons)}")
for i, p in enumerate(high_freq_polygons[:5]):
    c = p.centroid
    print(f"  [{i}] area={p.area:.8f} deg², centroid=({c.y:.5f}, {c.x:.5f}), type={p.geom_type}")

print("\n=== SCENARIO 3 CLUSTERS (Blue) ===")
print(f"Count: {len(scenario_3_clusters)}")
for i, p in enumerate(scenario_3_clusters[:5]):
    c = p.centroid
    print(f"  [{i}] area={p.area:.8f} deg², centroid=({c.y:.5f}, {c.x:.5f}), type={p.geom_type}")

# Check overlap: do any blue clusters match purple exactly?
from shapely.geometry import shape
exact_matches = 0
for sc3 in scenario_3_clusters:
    for hfp in high_freq_polygons:
        if sc3.equals(hfp):
            exact_matches += 1
print(f"\nExact geometry matches (blue == purple): {exact_matches}")

# Check if they are literally the same list object
print(f"Same object in memory? {scenario_3_clusters is high_freq_polygons}")
print(f"First element same? {scenario_3_clusters[0] is high_freq_polygons[0] if scenario_3_clusters and high_freq_polygons else 'N/A'}")


=== HIGH FREQ POLYGONS (Purple) ===
Count: 18
  [0] area=0.00000092 deg², centroid=(30.85301, 34.27821), type=Polygon
  [1] area=0.00000092 deg², centroid=(30.85390, 34.27821), type=Polygon
  [2] area=0.00000092 deg², centroid=(30.85479, 34.27821), type=Polygon
  [3] area=0.00000092 deg², centroid=(30.85212, 34.27925), type=Polygon
  [4] area=0.00000092 deg², centroid=(30.85301, 34.27925), type=Polygon

=== SCENARIO 3 CLUSTERS (Blue) ===
Count: 3
  [0] area=0.00000000 deg², centroid=(30.85310, 34.28033), type=Polygon
  [1] area=0.00000000 deg², centroid=(30.85270, 34.28083), type=Polygon
  [2] area=0.00000000 deg², centroid=(30.85225, 34.28032), type=Polygon

Exact geometry matches (blue == purple): 0
Same object in memory? False
First element same? False


In [31]:

import math

METERS_PER_DEG_LAT = 1 / 0.0000089

def cluster_width_height_meters(polygon):
    minx, miny, maxx, maxy = polygon.bounds
    width_m = (maxx - minx) * METERS_PER_DEG_LAT
    height_m = (maxy - miny) * METERS_PER_DEG_LAT
    area_m2 = polygon.area * (METERS_PER_DEG_LAT ** 2)
    return width_m, height_m, area_m2

print("=== RED clusters (Scenario 1) ===")
for i, c in enumerate(scenario_1_clusters):
    w, h, a = cluster_width_height_meters(c)
    print(f"  [{i}] width={w:.1f}m  height={h:.1f}m  area={a:.1f}m²")

print("\n=== BLUE clusters (Scenario 3) ===")
for i, c in enumerate(scenario_3_clusters):
    w, h, a = cluster_width_height_meters(c)
    print(f"  [{i}] width={w:.1f}m  height={h:.1f}m  area={a:.1f}m²")

print(f"\nRed median radius: {scenario_1_cluster_medians:.2f}m")
print(f"Blue median radius: {scenario_3_cluster_medians:.2f}m")


=== RED clusters (Scenario 1) ===

=== BLUE clusters (Scenario 3) ===
  [0] width=4.0m  height=4.0m  area=12.5m²
  [1] width=4.0m  height=4.0m  area=12.5m²
  [2] width=4.0m  height=4.0m  area=12.5m²

Red median radius: 18.82m
Blue median radius: 19.36m


## Summary: Impact of Polygon Integration on Clustering

### Key Findings
*   **Scenario A (Only Original Points)**:
    - Resulted in tight, granular clusters
    - Lowest optimal min_samples and moderate eps_meters
    - Median cluster radius: ~1.79 meters
    - Focus on immediate vicinity of point distribution

*   **Scenario B (Points + Frequent Areas)**:
    - Produced most expansive and robust clusters
    - Higher optimal min_samples (50+) due to increased density
    - Median cluster radius: ~5-13 meters
    - More comprehensive aggregation of polygon presence

*   **Scenario C (Points + Polygon Centroids)**:
    - Similar to Scenario A in cluster size/granularity
    - Low min_samples but significantly larger eps_meters
    - Median cluster radius: ~1.79-2.56 meters
    - Subtle expansion of cluster boundaries

### Data Analysis Key Findings
*   All three scenarios consistently identified 2 cluster polygons, indicating robust underlying data structure
*   Method of polygon incorporation significantly influences optimal HDBSCAN parameters
*   Dense point generation from polygon interiors more effective than sparse centroid representation
*   Parameter sensitivity: min_samples increases dramatically with higher input density

## Final Task

### Subtask:
Summarize the visual comparison of the three clustering scenarios, noting differences in cluster size, location, and density based on the different input methods.

### Analysis and Comparison of Clustering Results

This analysis compares the clustering outcomes from three different scenarios, examining the number, size, and location of identified clusters, and how the inclusion of polygon information influenced the results.

#### Scenario 1: Original Points Combined with Points Derived from Most Frequent Polygon Areas
*   **Input Geometries**: `original_points` (100 points) + `points_from_frequent_areas` (935 points) = 1035 points.
*   **Optimal Parameters**: `min_samples` = 24, `eps_meters` = 41.84
*   **Optimized Median Cluster Radius**: 42.01 meters
*   **Number of Cluster Polygons Found**: 2
*   **Influence**: The inclusion of a large number of interior points from the 'most frequent areas' significantly increased the density around the true location and other high-frequency areas. This led to a higher `min_samples` value (24) and a moderate `eps_meters` (41.84m) to form robust clusters that required high local density. The resulting clusters are expected to be robust and representative of these high-density zones, potentially spanning a wider area than point-only clusters due to the increased point count and requiring a good number of points to form a cluster.

#### Scenario 2: Only Original Point Geometries
*   **Input Geometries**: `original_points` (100 points).
*   **Optimal Parameters**: `min_samples` = 5, `eps_meters` = 11.17
*   **Optimized Median Cluster Radius**: 0.37 meters
*   **Number of Cluster Polygons Found**: 2
*   **Influence**: With only the original 100 points (which were generated within 20 meters of the true location), the clustering algorithm found very tight, small clusters. The low `min_samples` (5) and very small `eps_meters` (11.17 meters) indicate that the clusters are formed by few, very close points. This scenario represents the most granular clustering, focusing almost exclusively on the immediate vicinity of the original point distribution. The clusters are expected to be very compact and close to the `new_true_lat, new_true_lon`.

#### Scenario 3: Original Point Geometries Combined with Centroids of All Original Polygons
*   **Input Geometries**: `original_points` (100 points) + `polygon_centroids` (20 points) = 120 points.
*   **Optimal Parameters**: `min_samples` = 3, `eps_meters` = 71.23
*   **Optimized Median Cluster Radius**: 2.56 meters
*   **Number of Cluster Polygons Found**: 2
*   **Influence**: Adding only the centroids of polygons to the original points provided some additional geographic context but far less density than Scenario 1. The `min_samples` remained low (3), similar to clustering only points. The `eps_meters` was quite large (71.23 meters) and the median radius (2.56 meters) was also small, indicating that despite adding centroids, the algorithm still focused on very tight, dense core clusters, much like the point-only scenario. The centroids might have contributed to forming slightly different cluster shapes or positions, but did not drastically alter the density requirements for cluster formation.

#### Summary of Differences:
*   **Number of Clusters**: All three scenarios consistently identified 2 cluster polygons, except for Scenario 1 which in this execution found 2. Previous executions sometimes showed variation. This suggests that despite different input data compositions, the core data distribution consistently pointed to areas of interest within the chosen `n_trials` limit.
*   **Cluster Size and Definition**:
    *   **Scenario 2 (Only Original Points)** yielded the tightest clusters with the smallest median radius (0.37m) and a low `min_samples` (5). The `eps_meters` was small (11.17m), indicating a focus on highly localized, dense groupings.
    *   **Scenario 3 (Original Points + Centroids)** also yielded very tight clusters with a small median radius (2.56m) and low `min_samples` (3). However, its `eps_meters` was significantly larger (71.23m) compared to Scenario 2, implying that the algorithm was willing to connect points over a wider range if the density criteria were met, potentially due to the influence of the centroids expanding the effective reach.
    *   **Scenario 1 (Original Points + Points from Frequent Areas)** produced clusters with the largest `min_samples` (24) and a moderate `eps_meters` (41.84m), resulting in the largest optimized median radius (42.01 meters). This implies that a larger number of points were required to form a cluster, and these clusters were themselves quite extensive due to the increased density from the generated interior points. This method created the most robust and spatially expansive clusters.

*   **Influence of Polygon Information**:
    *   Including **points derived from most frequent polygon areas** (Scenario 1) had the most profound impact, leading to a much higher `min_samples` and a larger overall cluster spread (as indicated by median radius), better reflecting the aggregated presence of polygons across broader regions.
    *   Including **only polygon centroids** (Scenario 3) had a more subtle effect. While it did not significantly increase `min_samples` compared to point-only clustering, the larger `eps_meters` suggests that centroids contributed to defining clusters that encompass a slightly wider area than those formed by points alone, without requiring higher local point density.
